In [14]:
import numpy as np
import cv2

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)

NumPy: 1.26.4
OpenCV: 4.7.0


In [1]:
import cv2

tracker = cv2.legacy.TrackerCSRT_create()

print("Tracker Working")

Tracker Working


In [2]:
import cv2
import torch
import numpy as np
import pandas as pd

In [3]:
VIDEO="video2.mp4"

cap=cv2.VideoCapture(VIDEO)

print("Video Loaded:",cap.isOpened())

cap.release()

Video Loaded: True


In [4]:
#GenericObjectTracker

class GenericObjectTracker:

    def __init__(self):
        self.tracker = cv2.legacy.TrackerCSRT_create()

    def initialize(self, frame, bbox):
        self.tracker.init(frame, bbox)

    def track(self, frame):

        success, box = self.tracker.update(frame)

        x, y, w, h = map(int, box)

        return (x, y, w, h), 1.0

In [5]:
print(GenericObjectTracker)

<class '__main__.GenericObjectTracker'>


In [6]:
tracker = GenericObjectTracker()

print("Loaded")

Loaded


In [7]:
class GenericObjectTracker:

    def __init__(self):

        self.tracker = cv2.legacy.TrackerCSRT_create()

    def initialize(
        self,
        frame,
        bbox
    ):

        self.tracker.init(
            frame,
            bbox
        )

    def track(
        self,
        frame
    ):

        success, box = self.tracker.update(
            frame
        )

        x, y, w, h = map(
            int,
            box
        )

        frame_area = (
            frame.shape[0]
            *
            frame.shape[1]
        )

        object_area = max(
            w*h,
            1
        )

        tracker_conf = min(
            object_area
            /
            frame_area
            *
            20,
            1.0
        )

        if not success:
            tracker_conf = 0.0

        return (
            (
                x,
                y,
                w,
                h
            ),
            round(
                tracker_conf,
                3
            )
        )

In [8]:
#ConstantVelocityModel

class ConstantVelocityModel:

    def __init__(self):
        self.previous = None
        self.current = None

    def update(self, bbox):
        self.previous = self.current
        self.current = bbox

    def predict(self):

        if self.previous is None:
            return self.current, 0.0

        x1,y1,w1,h1 = self.previous
        x2,y2,w2,h2 = self.current

        vx = x2 - x1
        vy = y2 - y1

        speed = (vx**2 + vy**2)**0.5

        conf = max(
            0.3,
            1 - speed/100
        )

        prediction = (
            int(x2+vx),
            int(y2+vy),
            w2,
            h2
        )

        return prediction, round(conf,3)

In [9]:
#fusion

def combine(
    tracker_box,
    tracker_conf,
    motion_box,
    motion_conf
):

    total = tracker_conf + motion_conf

    if total == 0:
        return tracker_box, "Tracker"

    alpha = tracker_conf / total

    x = int(alpha*tracker_box[0] + (1-alpha)*motion_box[0])
    y = int(alpha*tracker_box[1] + (1-alpha)*motion_box[1])

    w = int(alpha*tracker_box[2] + (1-alpha)*motion_box[2])
    h = int(alpha*tracker_box[3] + (1-alpha)*motion_box[3])

    source = (
        "Tracker"
        if tracker_conf >= motion_conf
        else "Motion"
    )

    return (
        (x,y,w,h),
        source
    )

In [10]:
print(ConstantVelocityModel)
print(combine)

<class '__main__.ConstantVelocityModel'>
<function combine at 0x00000144A79E32E0>


In [11]:
VIDEO="video2.mp4"

print("Using Video:", VIDEO)

Using Video: video2.mp4


In [12]:
import time
import cv2
import pandas as pd

VIDEO="video2.mp4"

cap=cv2.VideoCapture(VIDEO)

ret,frame=cap.read()

if not ret:

    print("Video could not be loaded")

    cap.release()

    raise Exception()

roi_frame=cv2.resize(
    frame,
    (
        1280,
        720
    )
)

bbox=cv2.selectROI(
    "Select Object",
    roi_frame,
    False
)

scale_x=frame.shape[1]/1280
scale_y=frame.shape[0]/720

bbox=(
    int(bbox[0]*scale_x),
    int(bbox[1]*scale_y),
    int(bbox[2]*scale_x),
    int(bbox[3]*scale_y)
)

cv2.destroyAllWindows()

tracker=GenericObjectTracker()

tracker.initialize(
    frame,
    bbox
)

motion=ConstantVelocityModel()

motion.update(
    bbox
)

rows=[]

writer=cv2.VideoWriter(
    "tracked_video.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    30,
    (
        frame.shape[1],
        frame.shape[0]
    )
)

frame_id=0

while True:

    start=time.time()

    ret,frame=cap.read()

    if not ret:
        break

    t_box,t_conf=tracker.track(
        frame
    )

    m_box,m_conf=motion.predict()

    final_box,source=combine(
        t_box,
        t_conf,
        m_box,
        m_conf
    )

    motion.update(
        final_box
    )

    x,y,w,h=final_box

    cv2.rectangle(
        frame,
        (x,y),
        (x+w,y+h),
        (0,255,0),
        2
    )

    cv2.putText(
        frame,
        f"Tracker:{t_conf:.2f}",
        (20,30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0,255,0),
        2
    )

    cv2.putText(
        frame,
        f"Motion:{m_conf:.2f}",
        (20,60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255,0,0),
        2
    )

    cv2.putText(
        frame,
        f"Source:{source}",
        (20,90),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0,0,255),
        2
    )

    fps=1/(time.time()-start)

    cv2.putText(
        frame,
        f"FPS:{fps:.1f}",
        (20,120),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255,255,0),
        2
    )

    writer.write(
        frame
    )

    rows.append([
        frame_id,
        x,
        y,
        w,
        h,
        t_conf,
        m_conf,
        source
    ])

    frame_id+=1


cap.release()

writer.release()

pd.DataFrame(
    rows,
    columns=[
        "frame",
        "x",
        "y",
        "width",
        "height",
        "tracker_conf",
        "motion_conf",
        "trusted_source"
    ]
).to_csv(
    "tracking_output.csv",
    index=False
)

print("Completed")

Completed


In [13]:
import os

os.startfile("tracked_video.mp4")